In [1]:
import os
import psycopg2
import geopandas
import pandas as pd
from shapely.geometry import LineString

import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

con = psycopg2.connect(database="time", user="postgres", host="localhost")

In [2]:
# Read data
p1 = geopandas.read_postgis("select * from ilots_verniquet", con, geom_col='geom', index_col='gid')
p2 = geopandas.read_postgis("select * from ilots_vasserot", con, geom_col='geom', index_col='gid')
p3 = geopandas.read_postgis("select * from ilots_apur", con, geom_col='geom', index_col='gid')

In [3]:
def match(geom1, geom2, threshold1, threshold2):
    if geom1.intersects(geom2):
        g1 = geom1.intersection(geom2)
        area = g1.area
        return area > threshold1/2.0 and area > threshold2 * min(geom1.area, geom2.area)
    return False

def match_tables(table1, table2, threshold):
    min_area = min(table1['geom'].apply(lambda x: x.area).min(),table2['geom'].apply(lambda x: x.area).min())
    newdata = pd.DataFrame(columns = ['id', 'id1', 'id2', 'geometry'])
    count = 0
    for index1, row1 in table1.iterrows():
        g1 = row1['geom']
        if count == 1:
            break
        for index2, row2 in table2.iterrows():
            g2 = row2['geom']
            if match(g1, g2, min_area, threshold):
                line = LineString([g1.centroid,g2.centroid])
                newmatch = {'id':len(newdata), 'id1':index1, 'id2':index2, 'geometry':line}
                newdata.loc[len(newdata)] = newmatch
                if newmatch['id'] == 30:
                    count = 1
                    break
    return geopandas.GeoDataFrame(newdata, geometry='geometry')

In [4]:
ilots_stables_verniquet_vasserot = match_tables(p1, p2, 0.2)
ilots_stables_verniquet_vasserot.set_crs(p1.crs, inplace=True)

lots_stables_vasserot_apur = match_tables(p1, p3, 0.2)
lots_stables_vasserot_apur.set_crs(p1.crs, inplace=True)

ilots_stables_verniquet_apur = match_tables(p2, p3, 0.2)
ilots_stables_verniquet_apur.set_crs(p1.crs, inplace=True)

,id,id1,id2,geometry
0,0,3,2660,"LINESTRING (652308.932 6861458.904, 652285.813..."
1,1,4,3060,"LINESTRING (652237.685 6861489.774, 652241.055..."
2,2,5,3463,"LINESTRING (652209.067 6861523.481, 652203.055..."
3,3,6,2918,"LINESTRING (652071.187 6861576.381, 652071.852..."
4,4,7,2909,"LINESTRING (652108.571 6861531.263, 652110.743..."
5,5,8,2012,"LINESTRING (652153.868 6861485.464, 652173.002..."
6,6,8,3580,"LINESTRING (652153.868 6861485.464, 652129.803..."
7,7,9,241,"LINESTRING (649589.279 6861568.968, 649588.792..."
8,8,10,1795,"LINESTRING (652208.785 6861449.053, 652210.497..."
9,9,12,1411,"LINESTRING (652192.950 6861414.503, 652198.997..."


In [5]:
# Select distinct matches
ilots_stables_verniquet_vasserot.drop_duplicates(subset='id1', inplace=True)
ilots_stables_verniquet_vasserot.drop_duplicates(subset='id2', inplace=True)

lots_stables_vasserot_apur.drop_duplicates(subset='id1', inplace=True)
lots_stables_vasserot_apur.drop_duplicates(subset='id2', inplace=True)

ilots_stables_verniquet_apur.drop_duplicates(subset='id1', inplace=True)
ilots_stables_verniquet_apur.drop_duplicates(subset='id2', inplace=True)